# 09 - Cross-Validation Methodology

Implement rigorous validation strategies for soil prediction models.

## Validation Strategies
1. Spatial GroupKFold (prevent spatial leakage)
2. Leave-One-Field-Out (field generalization)
3. Temporal holdout (temporal generalization)
4. Stratified by soil type

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

DATA_DIR = Path('../../data/processed')
RESULTS_DIR = Path('../../results/evaluations')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Spatial GroupKFold

Prevents data leakage from spatial autocorrelation.

In [ ]:
def create_spatial_groups(df, n_groups=25):
    """Assign samples to spatial grid cells."""
    lat_bins = pd.cut(df['latitude'], bins=int(np.sqrt(n_groups)), labels=False)
    lon_bins = pd.cut(df['longitude'], bins=int(np.sqrt(n_groups)), labels=False)
    return lat_bins * int(np.sqrt(n_groups)) + lon_bins


def spatial_cross_validation(model, X, y, groups, n_splits=5):
    """
    Perform spatial cross-validation.
    
    Ensures validation folds are spatially separated from training.
    """
    gkf = GroupKFold(n_splits=n_splits)
    
    results = []
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        results.append({
            'fold': fold,
            'n_train': len(train_idx),
            'n_val': len(val_idx),
            'r2': r2_score(y_val, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_val, y_pred)),
            'mae': mean_absolute_error(y_val, y_pred)
        })
    
    return pd.DataFrame(results)

print("Spatial CV functions ready")

## 2. Leave-One-Field-Out

In [ ]:
def leave_one_field_out(model, X, y, field_ids):
    """
    Leave-one-field-out cross-validation.
    
    Tests generalization to completely unseen fields.
    """
    logo = LeaveOneGroupOut()
    
    results = []
    for train_idx, val_idx in logo.split(X, y, field_ids):
        field = field_ids.iloc[val_idx[0]]
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        results.append({
            'field': field,
            'n_samples': len(val_idx),
            'r2': r2_score(y_val, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_val, y_pred)),
            'mae': mean_absolute_error(y_val, y_pred)
        })
    
    return pd.DataFrame(results)

print("LOFO CV ready")

## 3. Visualization

In [ ]:
def plot_cv_results(results_df, title="Cross-Validation Results"):
    """Visualize CV results."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    metrics = ['r2', 'rmse', 'mae']
    for ax, metric in zip(axes, metrics):
        if 'fold' in results_df.columns:
            results_df.plot(x='fold', y=metric, kind='bar', ax=ax)
        else:
            results_df[metric].hist(ax=ax)
        
        ax.set_title(f'{metric.upper()}')
        ax.axhline(results_df[metric].mean(), color='r', linestyle='--', 
                  label=f'Mean: {results_df[metric].mean():.3f}')
        ax.legend()
    
    plt.suptitle(title)
    plt.tight_layout()
    return fig

## 4. Summary Statistics

In [ ]:
def summarize_cv_results(results_df, name="Model"):
    """Generate summary statistics."""
    summary = {
        'model': name,
        'n_folds': len(results_df),
        'r2_mean': results_df['r2'].mean(),
        'r2_std': results_df['r2'].std(),
        'rmse_mean': results_df['rmse'].mean(),
        'rmse_std': results_df['rmse'].std(),
        'mae_mean': results_df['mae'].mean(),
        'mae_std': results_df['mae'].std()
    }
    
    print(f"\n{name} Cross-Validation Summary:")
    print(f"  R² = {summary['r2_mean']:.3f} ± {summary['r2_std']:.3f}")
    print(f"  RMSE = {summary['rmse_mean']:.3f} ± {summary['rmse_std']:.3f}")
    print(f"  MAE = {summary['mae_mean']:.3f} ± {summary['mae_std']:.3f}")
    
    return summary

## 5. Recommendations

For soil nutrient prediction:

1. **Always use Spatial GroupKFold** - soil properties are spatially autocorrelated
2. **Report LOFO metrics** - shows true generalization to new fields
3. **Use overfitting ratio** - train_R²/val_R² should be < 1.5
4. **Multiple metrics** - R², RMSE, MAE capture different aspects